# Pipeline simple - Indicateur prix au m2 par quartier

Objectif: construire un indicateur de prix median au m2 exploitable au niveau quartier a partir du fichier DVF.

In [11]:
import pandas as pd
from pathlib import Path

# Notebook launched from pipeline/silver: remonte au root du projet
ROOT_DIR = Path.cwd().resolve().parents[1]
if not (ROOT_DIR / 'data').exists():
    ROOT_DIR = Path.cwd().resolve()
if not (ROOT_DIR / 'data').exists():
    raise FileNotFoundError("Impossible de localiser le dossier data du projet")

In [12]:
# 1) Chargement des donnees utiles
csv_path = ROOT_DIR / 'data' / 'bronze' / 'immobilier' / '75.csv'
use_cols = [
    'date_mutation',
    'valeur_fonciere',
    'surface_reelle_bati',
    'code_postal',
    'code_commune',
    'type_local'
]

df = pd.read_csv(csv_path, usecols=use_cols, parse_dates=['date_mutation'])

print(f'Nombre de lignes chargees: {len(df):,}'.replace(',', ' '))
print(f'Nombre de colonnes: {df.shape[1]}')
print('Apercu du jeu de donnees:')
display(df.head())
print('Types de biens (top 10):')
display(df['type_local'].value_counts(dropna=False).head(10))

Nombre de lignes chargees: 81 516
Nombre de colonnes: 6
Apercu du jeu de donnees:


,date_mutation,valeur_fonciere,code_postal,code_commune,type_local,surface_reelle_bati
0,2021-01-05,1480000.0,75008.0,75108,Appartement,111.0
1,2021-01-05,1480000.0,75008.0,75108,Dépendance,NaN
2,2021-01-05,1480000.0,75008.0,75108,Dépendance,NaN
3,2021-01-05,1480000.0,75008.0,75108,Dépendance,NaN
4,2021-01-05,20000.0,75008.0,75108,Dépendance,NaN


Types de biens (top 10):


type_local
Appartement                                 40885
Dépendance                                  34165
Local industriel. commercial ou assimilé     5420
NaN                                           824
Maison                                        222
Name: count, dtype: int64

## Comment lire la formule dans ce dataset DVF

On calcule le prix au m2 a partir de **deux colonnes du dataset**:

- `valeur_fonciere`: prix total de la transaction (en euros)
- `surface_reelle_bati`: surface du bien (en m2)

Formule appliquee ligne par ligne:

`prix_m2 = valeur_fonciere / surface_reelle_bati`

Exemple avec une ligne visible dans l'aperçu:

- `valeur_fonciere = 410000`
- `surface_reelle_bati = 31`
- `prix_m2 = 410000 / 31 = 13225.81 EUR/m2`

Pourquoi on nettoie avant ce calcul:

- on garde `Appartement` et `Maison` pour avoir un indicateur residentiel comparable

le KPI final agrège ce `prix_m2` par `annee` et `arrondissement`:

- `prix_m2_median`: valeur centrale (plus robuste aux extremes)
- `prix_m2_moyen`: moyenne (plus sensible aux biens tres chers)
- `nb_ventes`: nombre de transactions utilisees

In [13]:
# 2) Controle des valeurs manquantes puis nettoyage minimal
na_summary = df[['valeur_fonciere', 'surface_reelle_bati', 'date_mutation', 'code_postal']].isna().sum()
na_summary = na_summary.rename('nb_valeurs_manquantes').to_frame()
na_summary['pct_valeurs_manquantes'] = (na_summary['nb_valeurs_manquantes'] / len(df) * 100).round(2)

print('Valeurs manquantes par colonne (avant dropna):')
display(na_summary)


Valeurs manquantes par colonne (avant dropna):


,nb_valeurs_manquantes,pct_valeurs_manquantes
valeur_fonciere,1179,1.45
surface_reelle_bati,35012,42.95
date_mutation,0,0.00
code_postal,295,0.36


In [14]:
# 2bis) Nettoyage minimal avec suivi des volumes
n0 = len(df)

df = df.dropna(subset=['valeur_fonciere', 'surface_reelle_bati', 'date_mutation', 'code_postal'])
n1 = len(df)
print(f'Apres dropna: {n1:,} lignes ({(n1 / n0 * 100):.2f}% conservees)'.replace(',', ' '))

df = df[df['surface_reelle_bati'] > 9]  # evite les surfaces trop petites
n2 = len(df)
print(f'Apres filtre surface > 9: {n2:,} lignes'.replace(',', ' '))

df = df[df['valeur_fonciere'] > 10000]  # evite les valeurs anormales
n3 = len(df)
print(f'Apres filtre valeur_fonciere > 10000: {n3:,} lignes'.replace(',', ' '))

# Garde les biens residentiels les plus utiles
df = df[df['type_local'].isin(['Appartement', 'Maison'])]
n4 = len(df)
print(f"Apres filtre type_local (Appartement/Maison): {n4:,} lignes".replace(',', ' '))

# Convertit code postal en chaine pour extraire l'arrondissement
df['code_postal'] = df['code_postal'].astype(int).astype(str).str.zfill(5)
df = df[df['code_postal'].str.startswith('75')]
n5 = len(df)
print(f"Apres filtre code_postal commence par '75': {n5:,} lignes".replace(',', ' '))

print('Apercu apres nettoyage:')
display(df.head())

Apres dropna: 45 804 lignes (56.19% conservees)
Apres filtre surface > 9: 45 041 lignes
Apres filtre valeur_fonciere > 10000: 43 714 lignes
Apres filtre type_local (Appartement/Maison): 38 698 lignes
Apres filtre code_postal commence par '75': 38 698 lignes
Apercu apres nettoyage:


,date_mutation,valeur_fonciere,code_postal,code_commune,type_local,surface_reelle_bati
0,2021-01-05,1480000.0,75008,75108,Appartement,111.0
5,2021-01-08,410000.0,75001,75101,Appartement,31.0
8,2021-01-07,300000.0,75003,75103,Appartement,23.0
10,2021-01-06,1000000.0,75003,75103,Appartement,60.0
13,2021-01-08,1525559.0,75008,75108,Appartement,91.0


In [15]:
# 3) Calcul du prix au m2
df['prix_m2'] = df['valeur_fonciere'] / df['surface_reelle_bati']

print('Statistiques prix_m2 avant filtre outliers:')
display(df['prix_m2'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

# Filtre simple des outliers sur le prix au m2
q_low = df['prix_m2'].quantile(0.01)
q_high = df['prix_m2'].quantile(0.99)
print(f'Bornes outliers retenues: [{q_low:.2f}, {q_high:.2f}]')

n_before_outliers = len(df)
df = df[(df['prix_m2'] >= q_low) & (df['prix_m2'] <= q_high)]
n_after_outliers = len(df)
print(f'Apres filtre outliers: {n_after_outliers:,} lignes ({(n_after_outliers / n_before_outliers * 100):.2f}% conservees)'.replace(',', ' '))

print('Apercu prix_m2 apres filtre outliers:')
display(df[['valeur_fonciere', 'surface_reelle_bati', 'prix_m2']].head())

Statistiques prix_m2 avant filtre outliers:


count    3.869800e+04
mean     3.621991e+04
std      1.556171e+05
min      4.190871e+01
1%       6.020208e+02
5%       5.769231e+03
50%      1.126649e+04
95%      1.539689e+05
99%      4.761905e+05
max      7.090909e+06
Name: prix_m2, dtype: float64

Bornes outliers retenues: [602.02, 476190.48]
Apres filtre outliers: 37 925 lignes (98.00% conservees)
Apercu prix_m2 apres filtre outliers:


,valeur_fonciere,surface_reelle_bati,prix_m2
0,1480000.0,111.0,13333.333333
5,410000.0,31.0,13225.806452
8,300000.0,23.0,13043.478261
10,1000000.0,60.0,16666.666667
13,1525559.0,91.0,16764.384615


In [16]:
# 4) Construction des dimensions (annee, arrondissement)
df['annee'] = df['date_mutation'].dt.year
df['arrondissement'] = df['code_postal'].str[-2:]

# On garde les arrondissements classiques 01 a 20
arr_ok = [f'{i:02d}' for i in range(1, 21)]
n_before_arr = len(df)
df = df[df['arrondissement'].isin(arr_ok)]
n_after_arr = len(df)
print(f'Apres filtre arrondissements 01-20: {n_after_arr:,} lignes ({(n_after_arr / n_before_arr * 100):.2f}% conservees)'.replace(',', ' '))

print('Apercu dimensions:')
display(df[['annee', 'arrondissement', 'prix_m2']].head())

print('Nombre de ventes par annee:')
display(df['annee'].value_counts().sort_index())

print('Nombre de ventes par arrondissement:')
display(df['arrondissement'].value_counts().sort_index())

Apres filtre arrondissements 01-20: 37 925 lignes (100.00% conservees)
Apercu dimensions:


,annee,arrondissement,prix_m2
0,2021,08,13333.333333
5,2021,01,13225.806452
8,2021,03,13043.478261
10,2021,03,16666.666667
13,2021,08,16764.384615


Nombre de ventes par annee:


annee
2021    37925
Name: count, dtype: int64

Nombre de ventes par arrondissement:


arrondissement
01     370
02     593
03     781
04     561
05    1024
06     928
07    1118
08     799
09    1486
10    2012
11    3131
12    2001
13    1871
14    2011
15    4028
16    3139
17    3437
18    3942
19    2261
20    2432
Name: count, dtype: int64

In [17]:
# 5) Base intermediaire par arrondissement pour la projection quartier
kpi_base_projection = (
    df.groupby(['annee', 'arrondissement'], as_index=False)
      .agg(
          prix_m2_median=('prix_m2', 'median'),
          prix_m2_moyen=('prix_m2', 'mean'),
          nb_ventes=('prix_m2', 'size')
      )
      .sort_values(['annee', 'arrondissement'])
)

print(f'Nombre de lignes base intermediaire: {len(kpi_base_projection):,}'.replace(',', ' '))
print('Apercu base intermediaire (20 premieres lignes):')
display(kpi_base_projection.head(20))

print('Synthese base intermediaire (distributions):')
display(kpi_base_projection[['prix_m2_median', 'prix_m2_moyen', 'nb_ventes']].describe())

Nombre de lignes base intermediaire: 20
Apercu base intermediaire (20 premieres lignes):


,annee,arrondissement,prix_m2_median,prix_m2_moyen,nb_ventes
0,2021,01,14132.923077,43356.835182,370
1,2021,02,12352.941176,36549.151732,593
2,2021,03,13214.285714,36970.906432,781
3,2021,04,13534.482759,17053.540773,561
4,2021,05,13262.931034,27962.698282,1024
5,2021,06,15765.050167,22621.859649,928
6,2021,07,15277.777778,29788.978652,1118
7,2021,08,13529.411765,42608.098753,799
8,2021,09,12039.638932,25966.282623,1486
9,2021,10,10833.333333,19961.800635,2012


Synthese base intermediaire (distributions):


,prix_m2_median,prix_m2_moyen,nb_ventes
count,20.000000,20.000000,20.000000
mean,12088.212157,27435.813566,1896.250000
std,1756.405890,7543.659472,1155.341867
min,9609.756098,17053.540773,370.000000
25%,10769.972527,22425.570575,895.750000
50%,11831.919406,25795.276929,1936.000000
75%,13329.551217,30544.624228,2606.750000
max,15765.050167,43356.835182,4028.000000


In [ ]:
# 6) Initialisation export (sorties quartier uniquement)
output_dir = ROOT_DIR / 'data' / 'outputs'
output_dir.mkdir(exist_ok=True)

print('Export arrondissement desactive: seule la sortie quartier est conservee.')

Fichier cree: C:\Users\rayan\Desktop\efrei\projet-data-dev\data\outputs\kpi_prix_m2_arrondissement.csv
Nombre de lignes KPI: 20


## 7) Projection quartier pour la cartographie

La sortie finale est conservee au niveau quartier.

On utilise une aggregation intermediaire par arrondissement pour projeter les valeurs sur les quartiers:
- `prix_m2_median` et `prix_m2_moyen` sont repliques sur les quartiers de l'arrondissement
- `nb_ventes` est reparti selon la part de surface du quartier dans son arrondissement

In [18]:
quartiers_path = ROOT_DIR / 'data' / 'bronze' / 'commun' / 'quartier_paris.csv'
quartiers = pd.read_csv(quartiers_path, sep=';', encoding='utf-8-sig')

quartiers = quartiers.rename(
    columns={
        'C_QUINSEE': 'code_insee_quartier',
        'L_QU': 'nom_quartier',
        'C_AR': 'arrondissement_num',
        'SURFACE': 'surface_quartier_m2',
        'Geometry X Y': 'geometry_xy'
    }
)

quartiers['arrondissement'] = (
    pd.to_numeric(quartiers['arrondissement_num'], errors='coerce')
    .astype('Int64')
    .astype(str)
    .str.zfill(2)
)
quartiers['surface_quartier_m2'] = pd.to_numeric(quartiers['surface_quartier_m2'], errors='coerce')

surface_arr = (
    quartiers.groupby('arrondissement', as_index=False)['surface_quartier_m2']
    .sum()
    .rename(columns={'surface_quartier_m2': 'surface_arrondissement_m2'})
)

quartiers = quartiers.merge(surface_arr, on='arrondissement', how='left')
quartiers['part_surface'] = quartiers['surface_quartier_m2'] / quartiers['surface_arrondissement_m2']

kpi_prix_m2_quartier_estime = (
    kpi_base_projection.merge(
        quartiers[['code_insee_quartier', 'arrondissement', 'surface_quartier_m2', 'part_surface']],
        on='arrondissement',
        how='left'
    )
)

kpi_prix_m2_quartier_estime['nb_ventes_estime'] = (
    kpi_prix_m2_quartier_estime['nb_ventes'] * kpi_prix_m2_quartier_estime['part_surface']
).round().astype('Int64')

kpi_prix_m2_quartier_estime = kpi_prix_m2_quartier_estime[
    [
        'annee', 'arrondissement', 'code_insee_quartier',
        'prix_m2_median', 'prix_m2_moyen', 'nb_ventes', 'nb_ventes_estime',
        'surface_quartier_m2', 'part_surface'
    ]
] .sort_values(['annee', 'arrondissement', 'code_insee_quartier'])

print('Export intermediaire kpi_prix_m2_quartier_estime.csv desactive.')
print(f'Nombre de lignes quartier estime calculees: {len(kpi_prix_m2_quartier_estime):,}'.replace(',', ' '))
display(kpi_prix_m2_quartier_estime.head(12))

Export intermediaire kpi_prix_m2_quartier_estime.csv desactive.
Nombre de lignes quartier estime calculees: 80


,annee,arrondissement,code_insee_quartier,prix_m2_median,prix_m2_moyen,nb_ventes,nb_ventes_estime,surface_quartier_m2,part_surface
1,2021,01,7510101,14132.923077,43356.835182,370,176,869000.664564,0.476266
3,2021,01,7510102,14132.923077,43356.835182,370,84,412458.496330,0.226053
2,2021,01,7510103,14132.923077,43356.835182,370,56,273696.793301,0.150003
0,2021,01,7510104,14132.923077,43356.835182,370,55,269456.780599,0.147679
5,2021,02,7510201,12352.941176,36549.151732,593,112,188012.203850,0.189690
4,2021,02,7510202,12352.941176,36549.151732,593,146,243550.770623,0.245725
6,2021,02,7510203,12352.941176,36549.151732,593,166,278142.585084,0.280625
7,2021,02,7510204,12352.941176,36549.151732,593,168,281448.206577,0.283960
11,2021,03,7510301,13214.285714,36970.906432,781,212,318087.740454,0.271665
10,2021,03,7510302,13214.285714,36970.906432,781,181,271750.323937,0.232090


## 8) KPI unique prix au m2 par année et par quartier

Cette sortie fusionne les deux sources DVF (`75.csv` pour 2021 et `valeurs_foncieres.csv` pour 2025) pour produire une seule table exploitable par quartier.

In [ ]:
# 8) Calcul unifie du KPI prix au m2 par annee et quartier
from pathlib import Path
import pandas as pd

quartiers_path = ROOT_DIR / 'data' / 'bronze' / 'commun' / 'quartier_paris.csv'
quartiers_ref = pd.read_csv(quartiers_path, sep=';', encoding='utf-8-sig')
quartiers_ref = quartiers_ref.rename(
    columns={
        'C_QUINSEE': 'code_insee_quartier',
        'L_QU': 'nom_quartier',
        'C_AR': 'arrondissement_num',
        'SURFACE': 'surface_quartier_m2',
        'Geometry X Y': 'geometry_xy'
    }
)
quartiers_ref['arrondissement'] = pd.to_numeric(quartiers_ref['arrondissement_num'], errors='coerce').astype('Int64')
quartiers_ref['surface_quartier_m2'] = pd.to_numeric(quartiers_ref['surface_quartier_m2'], errors='coerce')
quartiers_ref = quartiers_ref.dropna(subset=['arrondissement', 'surface_quartier_m2']).copy()
quartiers_ref['arrondissement'] = quartiers_ref['arrondissement'].astype(int).astype(str).str.zfill(2)

surface_arr = (
    quartiers_ref.groupby('arrondissement', as_index=False)['surface_quartier_m2']
    .sum()
    .rename(columns={'surface_quartier_m2': 'surface_arrondissement_m2'})
)
quartiers_ref = quartiers_ref.merge(surface_arr, on='arrondissement', how='left')
quartiers_ref['part_surface'] = quartiers_ref['surface_quartier_m2'] / quartiers_ref['surface_arrondissement_m2']


def load_dvf_source(csv_file: Path) -> pd.DataFrame:
    use_cols_dvf = [
        'date_mutation',
        'valeur_fonciere',
        'surface_reelle_bati',
        'code_postal',
        'type_local',
    ]
    df_src = pd.read_csv(csv_file, usecols=use_cols_dvf, parse_dates=['date_mutation'])
    df_src = df_src.dropna(subset=['valeur_fonciere', 'surface_reelle_bati', 'date_mutation', 'code_postal'])
    df_src = df_src[df_src['surface_reelle_bati'] > 9]
    df_src = df_src[df_src['valeur_fonciere'] > 10000]
    df_src = df_src[df_src['type_local'].isin(['Appartement', 'Maison'])]
    df_src['code_postal'] = df_src['code_postal'].astype(int).astype(str).str.zfill(5)
    df_src = df_src[df_src['code_postal'].str.startswith('75')]
    df_src['prix_m2'] = df_src['valeur_fonciere'] / df_src['surface_reelle_bati']
    q_low = df_src['prix_m2'].quantile(0.01)
    q_high = df_src['prix_m2'].quantile(0.99)
    df_src = df_src[(df_src['prix_m2'] >= q_low) & (df_src['prix_m2'] <= q_high)]
    df_src['annee'] = df_src['date_mutation'].dt.year
    df_src['arrondissement'] = df_src['code_postal'].str[-2:]
    return df_src[['annee', 'arrondissement', 'prix_m2']].copy()


sources_dvf = [
    ROOT_DIR / 'data' / 'bronze' / 'immobilier' / '75.csv',
    ROOT_DIR / 'data' / 'bronze' / 'immobilier' / 'valeurs_foncieres.csv',
]

df_dvf_all = pd.concat([load_dvf_source(p) for p in sources_dvf], ignore_index=True)

def kpi_m2_q_annuel(df_in: pd.DataFrame) -> pd.DataFrame:
    out = (
        df_in.groupby(['annee', 'arrondissement'], as_index=False)
        .agg(
            prix_m2_median=('prix_m2', 'median'),
            prix_m2_moyen=('prix_m2', 'mean'),
            nb_ventes=('prix_m2', 'size')
        )
        .sort_values(['annee', 'arrondissement'])
    )
    out['arrondissement'] = out['arrondissement'].astype(str).str.zfill(2)
    return out

kpi_prix_m2_annuel = kpi_m2_q_annuel(df_dvf_all)

kpi_prix_m2_quartier_annuel = (
    kpi_prix_m2_annuel.merge(
        quartiers_ref[['code_insee_quartier', 'arrondissement', 'surface_quartier_m2', 'part_surface']],
        on='arrondissement',
        how='left'
    )
)
kpi_prix_m2_quartier_annuel['nb_ventes_estime'] = (
    kpi_prix_m2_quartier_annuel['nb_ventes'] * kpi_prix_m2_quartier_annuel['part_surface']
).round().astype('Int64')

kpi_prix_m2_quartier_annuel = kpi_prix_m2_quartier_annuel[
    [
        'annee', 'arrondissement', 'code_insee_quartier',
        'prix_m2_median', 'prix_m2_moyen', 'nb_ventes', 'nb_ventes_estime',
        'surface_quartier_m2', 'part_surface'
    ]
] .sort_values(['annee', 'arrondissement', 'code_insee_quartier'])

output_unifie = ROOT_DIR / 'data' / 'outputs' / 'kpi_prix_m2_quartier_annuel.csv'
kpi_prix_m2_quartier_annuel.to_csv(output_unifie, index=False)

print(f'Fichier cree: {output_unifie}')
print(f'Nombre de lignes unifiees: {len(kpi_prix_m2_quartier_annuel):,}'.replace(',', ' '))
display(kpi_prix_m2_quartier_annuel.head(12))

Fichier cree: C:\Users\rayan\Desktop\efrei\projet-data-dev\data\outputs\kpi_prix_m2_quartier_annuel.csv
Nombre de lignes unifiees: 160


,annee,arrondissement,code_insee_quartier,nom_quartier,prix_m2_median,prix_m2_moyen,nb_ventes,nb_ventes_estime,surface_quartier_m2,part_surface,geometry_xy
1,2021,01,7510101,Saint-Germain-l'Auxerrois,14132.923077,43356.835182,370,176,869000.664564,0.476266,"48.86065013520993, 2.3349103292801994"
3,2021,01,7510102,Halles,14132.923077,43356.835182,370,84,412458.496330,0.226053,"48.86228910809422, 2.3448988583110166"
2,2021,01,7510103,Palais-Royal,14132.923077,43356.835182,370,56,273696.793301,0.150003,"48.86465997810256, 2.3363089189653063"
0,2021,01,7510104,Place-Vendôme,14132.923077,43356.835182,370,55,269456.780599,0.147679,"48.867018590627694, 2.328581664931248"
5,2021,02,7510201,Gaillon,12352.941176,36549.151732,593,112,188012.203850,0.189690,"48.869306638088744, 2.3334318076581146"
4,2021,02,7510202,Vivienne,12352.941176,36549.151732,593,146,243550.770623,0.245725,"48.86910019977125, 2.3394607437483312"
6,2021,02,7510203,Mail,12352.941176,36549.151732,593,166,278142.585084,0.280625,"48.868008337402706, 2.3446991274274604"
7,2021,02,7510204,Bonne-Nouvelle,12352.941176,36549.151732,593,168,281448.206577,0.283960,"48.8671501183107, 2.3500801904088187"
11,2021,03,7510301,Arts-et-Métiers,13214.285714,36970.906432,781,212,318087.740454,0.271665,"48.86647028948414, 2.3570831310567835"
10,2021,03,7510302,Enfants-Rouges,13214.285714,36970.906432,781,181,271750.323937,0.232090,"48.86388739200144, 2.363123300986952"
